In [ ]:
from openplaces.api import read_entities
from openplaces.viz import create_street_view_link, show_building
from openplaces.geo.ids import decode_ubids

In [ ]:
admin_id = 'US-NC-BS'  # Brunswick, NC (CHEER pilot)

# Inspect

In [ ]:
parcels = read_entities('US-NC_parcel-nconemap-2025', admin_id, geom=True)
buildings_fema = read_entities('US_footprint-fema-2023', admin_id, geom=True)
buildings_nsi = read_entities('US_building-nsi-2022', admin_id, geom=True)
buildings_microsoft = read_entities('US_footprint-microsoft-v2', admin_id, geom=True)
if admin_id.startswith('US-NC'):
    buildings_nc = read_entities('US-NC_footprint-ncdps-2023', admin_id, geom=True)

In [ ]:
buildings_fema_sample = buildings_fema.sample(1)

In [ ]:
show_building(
    location=buildings_fema_sample,
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        'buildings_nc': buildings_nc,
    },
)

# FEMA footprint issues
## Non-existent

In [ ]:
BUILDING_ID = '8753XM25+MWC'

fig, ax = show_building(
    location=buildings_fema.loc[[BUILDING_ID]],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
    },
    # radius=600
    return_fig_ax=True,
)
ax.set_title('FEMA footprint without parcel')

# NSI issues
## Duplicate NSI points
All involve footprints other than Microsoft, FEMA, or Parcel

In [ ]:
mask_duplicate_ubid = buildings_nsi['building_id_ubid'].duplicated(keep=False)
buildings_nsi_duplicate_points = buildings_nsi[mask_duplicate_ubid].sort_values(
    ['building_id_ubid', 'source'], ascending=[True, False]
)
buildings_nsi_duplicate_points[
    ['building_id_ubid', 'purpose_group', 'purpose_subgroup', 'source']
]
buildings_nsi_duplicate_points['source'] = buildings_nsi_duplicate_points[
    'source'
].cat.rename_categories({'National Center for Education Statistics': 'NCES'})

In [ ]:
buildings_nsi_duplicate_points.groupby('building_id_ubid')['source'].apply(
    lambda x: ' + '.join(dict.fromkeys(x))
).value_counts()

In [ ]:
buildings_nsi_duplicate_sample = buildings_nsi_duplicate_points.sample()

fig, ax = show_building(
    # location=buildings_nsi_duplicate_points.iloc[:1],
    location=buildings_nsi_duplicate_sample,
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        # 'buildings_local': buildings_nc,
    },
    # radius=600
    return_fig_ax=True,
)
ax.set_title('Duplicate UBID')

In [ ]:
create_street_view_link(('9140 Forest Dr', 'Sunset Beach', 'NC'))

## Unique `ubid`, duplicate `openlocationcode`
NSI creates these because FEMA and Microsoft have different UBIDs for the same building

In [ ]:
mask_olc_duplicates = ~buildings_nsi['building_id_ubid'].duplicated(
    keep=False
) & buildings_nsi['building_id_ubid'].str.slice(0, 12).duplicated(keep=False)
buildings_need_ubid = buildings_nsi[mask_olc_duplicates][
    ['building_id_ubid', 'geometry']
].sort_values('building_id_ubid')
buildings_need_ubid.head()

In [ ]:
fig, ax = show_building(
    location=buildings_need_ubid.iloc[:1],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        # 'buildings_local': buildings_nc,
    },
    # radius=600
    return_fig_ax=True,
)
buildings_need_ubid.iloc[:2].to_crs('epsg:3857').plot(ax=ax, color='red')
ubid_bboxes = decode_ubids(buildings_need_ubid['building_id_ubid'].iloc[:2]).to_crs(
    'epsg:3857'
)
ubid_bboxes.boundary.plot(ax=ax, color='red', alpha=0.5)
ax.set_title(
    'Unique UBID (bounding box: red), duplicate `openlocationcode` (coarse point)'
)

In [ ]:
fig, ax = show_building(
    location=buildings_need_ubid.iloc[2:3],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        # 'buildings_local': buildings_nc,
    },
    # radius=600
    return_fig_ax=True,
)
buildings_need_ubid.iloc[2:4].to_crs('epsg:3857').plot(ax=ax, color='red')
ubid_bboxes = decode_ubids(buildings_need_ubid['building_id_ubid'].iloc[2:4]).to_crs(
    'epsg:3857'
)
ubid_bboxes.boundary.plot(ax=ax, color='red', alpha=0.5)
ax.set_title(
    'Unique UBID (bounding box: red), duplicate `openlocationcode` (coarse point)'
)

In [ ]:
fig, ax = show_building(
    location=buildings_need_ubid.iloc[4:5],
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        # 'buildings_local': buildings_nc,
    },
    # radius=600
    return_fig_ax=True,
)
buildings_need_ubid.iloc[4:6].to_crs('epsg:3857').plot(ax=ax, color='red')
ubid_bboxes = decode_ubids(buildings_need_ubid['building_id_ubid'].iloc[4:6]).to_crs(
    'epsg:3857'
)
ubid_bboxes.boundary.plot(ax=ax, color='red', alpha=0.5)
ax.set_title(
    'Unique UBID (bounding box: red), duplicate `openlocationcode` (coarse point)'
)

## Where does the UBID come from?

In [ ]:
import numpy as np
from openplaces.geo.ids import get_ubids

buildings_fema['ubid'] = get_ubids(buildings_fema)
buildings_microsoft['ubid'] = get_ubids(buildings_microsoft)
buildings_nsi['ubid_origin'] = np.where(
    buildings_nsi['building_id_ubid'].isin(buildings_microsoft['ubid']),
    np.where(
        buildings_nsi['building_id_ubid'].isin(buildings_fema['ubid']),
        'both',
        'microsoft',
    ),
    np.where(
        buildings_nsi['building_id_ubid'].isin(buildings_fema['ubid']),
        'fema',
        'unknown',
    ),
)

buildings_nsi['ubid_origin'].value_counts()

In [ ]:
# UBID_ORIGIN = 'fema'
# UBID_ORIGIN = 'microsoft'
UBID_ORIGIN = 'unknown'

buildings_nsi_sample = buildings_nsi[
    buildings_nsi['ubid_origin'].eq(UBID_ORIGIN)
].sample()
fig, ax = show_building(
    location=buildings_nsi_sample,
    geodatasets={
        'parcels': parcels,
        'buildings_fema': buildings_fema,
        'buildings_nsi': buildings_nsi,
        'buildings_microsoft': buildings_microsoft,
        'buildings_local': buildings_nc,
    },
    # radius=600
    return_fig_ax=True,
)
ax.set_title(f'Example: NSI origin = {UBID_ORIGIN}')